## Time Charging

Imagine that your mobile phone is currently out of battery, but you need to use it for `t` more minutes. Luckily, you have a number of spare batteries that are fully charged, where the `i`th battery lets you use the phone for `capacity[i]` more minutes. After `capacity[i]`, this battery becomes depleted, and has to be fully recharged for `recharge[i]` minutes before you can use it again. You use the extra batteries in the given order until each one is fully depleted, and then switch to the next one. If the next battery is still recharging, skip it and try the next one. This process continues cyclically until you are done using your phone after `t` minutes.

Return the number of full batteries used during the `t` minutes you need to use your phone. If it is impossible to have the phone working during the entire duration of `t` minutes — i.e., if at some point all batteries are recharging and unavailable, return `-1`.

**Note:** You are not expected to provide the most optimal solution, but a solution with time complexity not worse than **O(t · capacity.length)** will fit within the execution time limit.

### Example

- For `t = 16`, `capacity = [2, 5, 6]`, and `recharge = [12, 1, 4]`, the output should be `solution(t, capacity, recharge) = 3`.
  - At `t = 0`: Battery 0 with `capacity[0] = 2`, phone usable for 2 minutes.
  - At `t = 2`: Battery 0 depleted (recharged at `t = 14`), switch to battery 1 with `capacity[1] = 5`, 5 more minutes.
  - At `t = 7`: Battery 1 depleted (recharged at `t = 8`), switch to battery 2 with `capacity[2] = 6`, 6 more minutes.
  - At `t = 13`: Battery 2 depleted (recharged at `t = 17`), try battery 0 — still recharging until `t = 14`. Battery 1 is ready, use it for 5 more minutes.
  - Continue on battery 1 until `t = 16`. Total: **3** full batteries used.

- For `t = 16`, `capacity = [2, 5, 6]`, and `recharge = [12, 8, 4]`, the output should be `solution(t, capacity, recharge) = -1`.
  - At `t = 13`: Battery 2 depleted. Battery 0 recharging until `t = 14`, battery 1 until `t = 15`, battery 2 until `t = 17`. All unavailable → return `-1`.

### Input/Output

- **[execution time limit]** 4 seconds (py3)
- **[memory limit]** 1 GB
- **[input]** `integer` `t` — The amount of time you need to use your phone. Guaranteed constraints: `1 ≤ t ≤ 5000`
- **[input]** `array.integer` `capacity` — Number of minutes each battery provides. Guaranteed constraints: `1 ≤ capacity.length ≤ 100`, `1 ≤ capacity[i] < 100`
- **[input]** `array.integer` `recharge` — Minutes required to fully recharge each battery. Guaranteed constraints: `1 ≤ recharge.length ≤ 100`, `recharge.length = capacity.length`, `1 ≤ recharge[i] < 100`
- **[output]** `integer` — Number of full batteries used, or `-1` if impossible.

In [48]:
import heapq

def solution(t, capacity, recharge):

    # Think of it like a deli counter — each battery takes a number ticket
    # Lowest ticket = next to be served. When a battery comes back from charging
    # it gets a FRESH ticket — it rejoins at the back of the line
    ticket_counter = 0

    # ready_line: (ticket, battery) — lowest ticket gets picked next
    ready_line = []
    for battery in range(len(capacity)):
        heapq.heappush(ready_line, (ticket_counter, battery))
        ticket_counter += 1

    # charger: (done_at, ticket, battery) — earliest done floats to the top
    # ticket is stamped when the battery is SENT away — so it knows its place in line when it returns
    charger = []

    clock         = 0
    fully_drained = 0

    while clock < t:

        # Graduate any battery that finished charging — it rejoins with its original ticket
        while charger and charger[0][0] <= clock:
            _, ticket, battery = heapq.heappop(charger)
            heapq.heappush(ready_line, (ticket, battery))

        if not ready_line:
            return -1  # everyone still charging — no way forward

        # Serve the next battery in line (lowest ticket = been waiting longest = cyclic order)
        _, battery = heapq.heappop(ready_line)
        fully_drained += 1
        clock         += capacity[battery]

        # Battery just died — stamp it a fresh ticket and send it to the charger
        heapq.heappush(charger, (clock + recharge[battery], ticket_counter, battery))
        ticket_counter += 1

    # Last battery wasn't fully drained — phone hit t mid-charge, doesn't count
    if clock > t:
        fully_drained -= 1

    return fully_drained


assert solution(16, [2, 5, 6], [12, 1, 4]) == 3
assert solution(16, [2, 5, 6], [12, 8, 4]) == -1
assert solution(1,  [1],       [99])       == 1
assert solution(2,  [1],       [99])       == -1
assert solution(5,  [10],      [1])        == 0
assert solution(10, [5, 3],    [1, 2])    == 2
assert solution(4,  [2, 2],    [1, 1])    == 2
assert solution(3,  [1, 1],    [5, 5])    == -1
assert solution(20, [2, 1, 10],[1, 10, 3]) == 5
print('all pass')

all pass


In [47]:
from collections import deque

def solution(t, capacity, recharge):

    # ready_at is the charger — each battery stamps when it'll be done
    # no separate structure needed, the deque rotation handles everything
    ready_at   = [0] * len(capacity)
    ready_line = deque(range(len(capacity)))  # everyone starts ready, in cyclic order

    clock         = 0
    fully_drained = 0

    while clock < t:
        found = False
        for _ in range(len(ready_line)):
            battery = ready_line.popleft()

            if ready_at[battery] <= clock:               # off the charger?
                fully_drained += 1
                clock         += capacity[battery]
                ready_at[battery] = clock + recharge[battery]  # stamp when it's back
                ready_line.append(battery)               # just used — back of the line
                found = True
                break
            else:
                ready_line.append(battery)               # still charging — rotate past

        if not found:
            return -1

    if clock > t: fully_drained -= 1  # last battery wasn't fully drained — doesn't count
    return fully_drained


assert solution(16, [2, 5, 6], [12, 1, 4]) == 3
assert solution(16, [2, 5, 6], [12, 8, 4]) == -1
assert solution(1,  [1],       [99])       == 1
assert solution(2,  [1],       [99])       == -1
assert solution(5,  [10],      [1])        == 0
assert solution(10, [5, 3],    [1, 2])    == 2
assert solution(4,  [2, 2],    [1, 1])    == 2
assert solution(3,  [1, 1],    [5, 5])    == -1
assert solution(20, [2, 1, 10],[1, 10, 3]) == 5
print('all pass')

all pass
